# Lab 11: Multiomic Analysis using Cooperative Logistic Regression

This notebook is a starter template for implementing the cooperative logistic regression method described in the below paper. The approach is designed for integrating multiple data modalities (such as imaging and gene expression) in a unified predictive model. The cooperative logistic regression framework encourages the models built on each modality to produce similar predictions, while also allowing for modality-specific effects.

For theoretical background and mathematical details and further applications of this method, see: [Cooperative learning for multiview analysis](https://arxiv.org/abs/2112.12337)

**What you'll find in this notebook:**
- Simulated data representing two modalities (e.g., imaging and genomics)
- Implementation of the cooperative logistic regression loss function
- Model fitting using numerical optimization
- Evaluation metrics for binary classification

**Next steps:**  
You can adapt this notebook to your own datasets and extend it to include additional analyses, visualizations, or comparisons with other integration strategies.

In [ ]:
# IMPORTS
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, log_loss
)
from scipy.optimize import minimize
from scipy.special import expit # this is the sigmoid function

In [ ]:
# DATA LOADING AND PREPROCESSING

np.random.seed(0)
n_samples = 500
n_img_features = 30
n_gene_features = 100

# Imaging and gene expression features
X_img = np.random.randn(n_samples, n_img_features)
X_gene = np.random.randn(n_samples, n_gene_features)

# Simulated binary outcome (e.g. histology from EHR)
y = np.random.randint(0, 2, size=n_samples)

# Train/test split
X_img_train, X_img_test, X_gene_train, X_gene_test, y_train, y_test = train_test_split(
    X_img, X_gene, y, test_size=0.3, random_state=42
)

# Standardize features
scaler_img = StandardScaler().fit(X_img_train)
scaler_gene = StandardScaler().fit(X_gene_train)

X_img_train_std = scaler_img.transform(X_img_train)
X_gene_train_std = scaler_gene.transform(X_gene_train)
X_img_test_std = scaler_img.transform(X_img_test)
X_gene_test_std = scaler_gene.transform(X_gene_test)

## TO DO: Data Loading and Preparation (NSCLC Radiogenomics)

After reviewing the simulated example, you can proceed with the following steps using the NSCLC Radiogenomics dataset from TCIA (available on Blackboard):

1. **Load the Dataset**
      - Import the NSCLC Radiogenomics dataset into your notebook.

2. **Explore the Data**
      - Inspect the first few rows and columns.
      - Check for missing values and data types.
      - Summarize key statistics for each modality (e.g., imaging features, genomics).

3. **Format the Data**
      - Encode categorical variables if needed.
      - Standardize or normalize features as appropriate.

4. **Define the Prediction Task**
      - Identify the target variable (e.g., clinical data such as mutation status or histology).

In [ ]:
# MODEL DEFINITION / METHODS (NO NEED TO EDIT)

# ----- Cooperative logistic regression loss function -----
def cooperative_logistic_loss(params, X1, X2, y, rho, lambda1, lambda2):
    p1 = X1.shape[1]
    theta1 = params[:p1]
    theta2 = params[p1:]
    logits = X1 @ theta1 + X2 @ theta2
    prob = expit(logits)
    
    log_likelihood = -np.sum(y * np.log(prob + 1e-15) + (1 - y) * np.log(1 - prob + 1e-15))
    coop_penalty = rho / 2 * np.sum((X1 @ theta1 - X2 @ theta2) ** 2)
    l1_penalty = lambda1 * np.sum(np.abs(theta1)) + lambda2 * np.sum(np.abs(theta2))
    
    return log_likelihood + coop_penalty + l1_penalty

# ----- Model fitting -----
def fit_cooperative_logistic(X1, X2, y, rho=1.0, lambda1=0.1, lambda2=0.1):
    p1 = X1.shape[1]
    p2 = X2.shape[1]
    init_params = np.zeros(p1 + p2)

    result = minimize(
        cooperative_logistic_loss,
        init_params,
        args=(X1, X2, y, rho, lambda1, lambda2),
        method='L-BFGS-B'
    )

    theta1 = result.x[:p1]
    theta2 = result.x[p1:]
    return theta1, theta2

In [ ]:
# MODEL TRAINING AND EVALUATION

# ----- Train model and evaluate -----
theta_img, theta_gene = fit_cooperative_logistic(
    X_img_train_std, X_gene_train_std, y_train, rho=1.0, lambda1=0.1, lambda2=0.1
)

logits_test = X_img_test_std @ theta_img + X_gene_test_std @ theta_gene
probs_test = expit(logits_test)
y_pred = (probs_test >= 0.5).astype(int)

# ----- Evaluation -----
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "ROC AUC": roc_auc_score(y_test, probs_test),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "Log Loss": log_loss(y_test, probs_test)
}

print(metrics)

## Further Tasks and Extensions

Now that you have a working cooperative logistic regression implementation, consider the following tasks to deepen your analysis with the dataset:

- **Hyperparameter Search:**  
  Loop over a range of values for `rho`, `lambda1`, and `lambda2`. For each combination, fit the model and record the evaluation metrics (e.g., accuracy, ROC AUC, log loss). Concatenate the results into a DataFrame for easy comparison and visualization.

- **Hyperparameter Optimization (Advanced):**  
  Instead of a grid search, use an optimization algorithm (e.g., Bayesian optimization or randomized search) to efficiently find the optimal values of `rho` and `lambda` that maximize model performance.

- **Target Variable Exploration:**  
  Try using different target variables (e.g., mutation status, histology, survival outcome) and repeat the hyperparameter search. Investigate whether the optimal value of `rho` changes depending on the target, and discuss what this might mean biologically or clinically.

- **Modality Importance:**  
  Analyze the learned coefficients (`theta_img`, `theta_gene`) for each modality. Are certain modalities more predictive for specific targets? Visualize the coefficient magnitudes.

- **Comparison with Standard Logistic Regression:**  
  Fit a standard logistic regression model using only one modality or by concatenating both modalities. Compare its performance to the cooperative approach.

- **Visualization:**  
  Create heatmaps or line plots to visualize how performance metrics vary with `rho` and `lambda`. This can help identify trends and optimal regions in the hyperparameter space.

- **Robustness Checks:**  
  Assess the stability of your results by repeating the analysis with different random seeds or train/test splits.